In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv


In [2]:
import pandas as pd
import numpy as np

In [ ]:
train=pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [4]:
print("Train shape ",  train.shape)
print("Test shape ", test.shape)

Train shape  (8693, 14)
Test shape  (4277, 13)


In [5]:
display(train.head())
train.info()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [6]:
missing = train.isnull().sum()
missing_pct =(train.isnull().mean()*100).round(2)

In [7]:
missing_table =pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
display(missing_table.sort_values('Missing Count' , ascending =False))


,Missing Count,Missing %
CryoSleep,217,2.50
ShoppingMall,208,2.39
VIP,203,2.34
HomePlanet,201,2.31
Name,200,2.30
Cabin,199,2.29
VRDeck,188,2.16
Spa,183,2.11
FoodCourt,183,2.11
Destination,182,2.09


In [8]:
print("Duplicate rows :", train.duplicated().sum())
print("Duplicate Passenger ID:" ,train['PassengerId'].duplicated().sum())

Duplicate rows : 0
Duplicate Passenger ID: 0


In [9]:
print(train['Transported'].value_counts())
print(train['Transported'].value_counts(normalize = True).round(3) * 100)

Transported
True     4378
False    4315
Name: count, dtype: int64
Transported
True     50.4
False    49.6
Name: proportion, dtype: float64


In [10]:
display(train.describe())

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8514.000000,8512.000000,8510.000000,8485.000000,8510.000000,8505.000000
mean,28.827930,224.687617,458.077203,173.729169,311.138778,304.854791
std,14.489021,666.717663,1611.489240,604.696458,1136.705535,1145.717189
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,47.000000,76.000000,27.000000,59.000000,46.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [11]:
spend_cols = ['RoomService' , 'FoodCourt', 'ShoppingMall' ,'Spa','VRDeck']
total_spend = train[spend_cols].sum(axis = 1)
print("Zero spend passengers %:",round((total_spend == 0).mean() *100, 1))
print(total_spend.groupby(train['CryoSleep']).agg(['mean','median']).round(1))

Zero spend passengers %: 42.0
             mean  median
CryoSleep                
False      2248.3  1019.0
True          0.0     0.0


In [12]:
print(train['CryoSleep'].value_counts(dropna = False))
print(train.groupby('CryoSleep')['Transported'].mean().round(3))


CryoSleep
False    5439
True     3037
NaN       217
Name: count, dtype: int64
CryoSleep
False    0.329
True     0.818
Name: Transported, dtype: float64


In [13]:
print("HomePlanet: \n",train.groupby('HomePlanet')['Transported'].mean().round(3))
print("\nDestination: \n",train.groupby('Destination')['Transported'].mean().round(3))
print("\nVIP:\n",train.groupby('VIP')['Transported'].mean().round(3))

HomePlanet: 
 HomePlanet
Earth     0.424
Europa    0.659
Mars      0.523
Name: Transported, dtype: float64

Destination: 
 Destination
55 Cancri e      0.610
PSO J318.5-22    0.504
TRAPPIST-1e      0.471
Name: Transported, dtype: float64

VIP:
 VIP
False    0.506
True     0.382
Name: Transported, dtype: float64


In [14]:
age_bins = pd.cut(train['Age'],bins =[0,12,18,30,50,79],labels=['Child(0-12)','Teen(13-18)','YoungAdult(19-30)','Adult(31-50)','Senior(51-79)'])
print(train.groupby(age_bins,observed = True)['Transported'].mean().round(3))

Age
Child(0-12)          0.669
Teen(13-18)          0.537
YoungAdult(19-30)    0.468
Adult(31-50)         0.479
Senior(51-79)        0.484
Name: Transported, dtype: float64


In [15]:
datasets = [train,test]
spend_cols =['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
for df in datasets:
    cryo_true_mask = df['CryoSleep'] == True
    for col in spend_cols:
        df.loc[cryo_true_mask,col] = df.loc[cryo_true_mask,col].fillna(0)
    total_spend = df[spend_cols].sum(axis = 1)
    spend_positive_mask = (total_spend > 0) & (df['CryoSleep'].isna())
    df.loc[spend_positive_mask,'CryoSleep'] = False
print("Train CryoSleep missing ab:", train['CryoSleep'].isna().sum())
print("Train spending missing ab : \n",train[spend_cols].isna().sum())

Train CryoSleep missing ab: 98
Train spending missing ab : 
 RoomService     113
FoodCourt       113
ShoppingMall    112
Spa             118
VRDeck          126
dtype: int64


In [16]:
for df in datasets:
    df['CryoSleep']=df['CryoSleep'].fillna(False)
    false_group_median = train.loc[train['CryoSleep']== False,spend_cols].median()
    print("CryoSleep False group median (for reference purpose): \n", false_group_median)
    for col in spend_cols:
        df[col] = df[col].fillna(false_group_median[col])
        
print("\n Train CryoSleep missing ab :",train['CryoSleep'].isna().sum()) 
print("Train spending missing ab: \n",train[spend_cols].isna().sum())

CryoSleep False group median (for reference purpose): 
 RoomService     2.0
FoodCourt       4.0
ShoppingMall    1.0
Spa             6.0
VRDeck          3.0
dtype: float64
CryoSleep False group median (for reference purpose): 
 RoomService     2.0
FoodCourt       4.0
ShoppingMall    1.0
Spa             6.0
VRDeck          3.0
dtype: float64

 Train CryoSleep missing ab : 0
Train spending missing ab: 
 RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
dtype: int64


/tmp/ipykernel_16/4055585206.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['CryoSleep']=df['CryoSleep'].fillna(False)
/tmp/ipykernel_16/4055585206.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['CryoSleep']=df['CryoSleep'].fillna(False)


In [17]:
print("Train missing ab :\n",train.isnull().sum())
print("\n Test missing ab : \n",test.isnull().sum())

Train missing ab :
 PassengerId       0
HomePlanet      201
CryoSleep         0
Cabin           199
Destination     182
Age             179
VIP             203
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Name            200
Transported       0
dtype: int64

 Test missing ab : 
 PassengerId       0
HomePlanet       87
CryoSleep         0
Cabin           100
Destination      92
Age              91
VIP              93
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Name             94
dtype: int64


In [18]:
simple_cols = ['HomePlanet', 'Destination', 'VIP']

for df in datasets:
    for col in simple_cols:
        mode_value = train[col].mode()[0]   # train se mode nikal rahe hain (leakage se bachne ke liye)
        df[col] = df[col].fillna(mode_value)
    
    median_age = train['Age'].median()      # train se median
    df['Age'] = df['Age'].fillna(median_age)

# Check
print("Train missing ab:\n", train.isnull().sum())
print("\nTest missing ab:\n", test.isnull().sum())

Train missing ab:
 PassengerId       0
HomePlanet        0
CryoSleep         0
Cabin           199
Destination       0
Age               0
VIP               0
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Name            200
Transported       0
dtype: int64

Test missing ab:
 PassengerId       0
HomePlanet        0
CryoSleep         0
Cabin           100
Destination       0
Age               0
VIP               0
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Name             94
dtype: int64


/tmp/ipykernel_16/1302717978.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(mode_value)


In [19]:
for df in datasets:
    
    cabin_split = df['Cabin'].str.split('/', expand=True)
    df['Deck'] = cabin_split[0]
    df['CabinNum'] = cabin_split[1]
    df['Side'] = cabin_split[2]


display(train[['Cabin', 'Deck', 'CabinNum', 'Side']].head())


print(train[['Deck', 'CabinNum', 'Side']].isnull().sum())

,Cabin,Deck,CabinNum,Side
0,B/0/P,B,0,P
1,F/0/S,F,0,S
2,A/0/S,A,0,S
3,A/0/S,A,0,S
4,F/1/S,F,1,S


Deck        199
CabinNum    199
Side        199
dtype: int64


In [20]:
for df in datasets:
    
    df['CabinNum'] = pd.to_numeric(df['CabinNum'])

   
    for col in ['Deck', 'Side']:
        mode_value = train[col].mode()[0]
        df[col] = df[col].fillna(mode_value)

  
    median_cabinnum = train['CabinNum'].median()
    df['CabinNum'] = df['CabinNum'].fillna(median_cabinnum)


print("Train missing ab:\n", train[['Deck', 'CabinNum', 'Side']].isnull().sum())
print("\nTest missing ab:\n", test[['Deck', 'CabinNum', 'Side']].isnull().sum())

Train missing ab:
 Deck        0
CabinNum    0
Side        0
dtype: int64

Test missing ab:
 Deck        0
CabinNum    0
Side        0
dtype: int64


In [21]:
for df in datasets:
  
    df['Group'] = df['PassengerId'].str.split('_').str[0]
    
    
    group_sizes = df['Group'].value_counts()
    df['GroupSize'] = df['Group'].map(group_sizes)
    
    
    df['IsSolo'] = (df['GroupSize'] == 1)
    
    
    df['Surname'] = df['Name'].fillna('Unknown Unknown').str.split(' ').str[-1]


display(train[['PassengerId', 'Group', 'GroupSize', 'IsSolo', 'Name', 'Surname']].head())
print("\nGroupSize distribution:\n", train['GroupSize'].value_counts().sort_index())

,PassengerId,Group,GroupSize,IsSolo,Name,Surname
0,0001_01,0001,1,True,Maham Ofracculy,Ofracculy
1,0002_01,0002,1,True,Juanna Vines,Vines
2,0003_01,0003,2,False,Altark Susent,Susent
3,0003_02,0003,2,False,Solam Susent,Susent
4,0004_01,0004,1,True,Willy Santantines,Santantines



GroupSize distribution:
 GroupSize
1    4805
2    1682
3    1020
4     412
5     265
6     174
7     231
8     104
Name: count, dtype: int64


In [22]:
for df in datasets:
    df.drop(columns=['Cabin','Name','Group'],inplace = True)
    df['CryoSleep']=df['CryoSleep'].astype(bool)
    df['VIP']=df['VIP'].astype(bool)
print(train.dtypes)    
    
    


PassengerId      object
HomePlanet       object
CryoSleep          bool
Destination      object
Age             float64
VIP                bool
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Transported        bool
Deck             object
CabinNum        float64
Side             object
GroupSize         int64
IsSolo             bool
Surname          object
dtype: object


In [23]:
# Surname ko abhi drop kar dete hain (baseline ke liye)
for df in datasets:
    df.drop(columns=['Surname'], inplace=True)

# One-hot encoding categorical columns ke liye
categorical_cols = ['HomePlanet', 'Destination', 'Deck', 'Side']

train_encoded = pd.get_dummies(train, columns=categorical_cols, drop_first=True)
test_encoded = pd.get_dummies(test, columns=categorical_cols, drop_first=True)

print("Train shape before:", train.shape, "-> after:", train_encoded.shape)
print("\nNaye columns (encoded):\n", [col for col in train_encoded.columns if col not in train.columns])

Train shape before: (8693, 17) -> after: (8693, 25)

Naye columns (encoded):
 ['HomePlanet_Europa', 'HomePlanet_Mars', 'Destination_PSO J318.5-22', 'Destination_TRAPPIST-1e', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T', 'Side_S']


In [24]:
from sklearn.model_selection import train_test_split

X = train_encoded.drop(columns=['PassengerId','Transported'])
y = train_encoded['Transported']

X_train,X_val,y_train,y_val = train_test_split(
    X,y,test_size = 0.2,random_state = 42,stratify = y
)
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("\nTrain target balance:\n", y_train.value_counts(normalize=True).round(3))
print("\nVal target balance:\n", y_val.value_counts(normalize=True).round(3))


X_train shape: (6954, 23)
X_val shape: (1739, 23)

Train target balance:
 Transported
True     0.504
False    0.496
Name: proportion, dtype: float64

Val target balance:
 Transported
True     0.504
False    0.496
Name: proportion, dtype: float64


In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Model banao aur train karo
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Validation set pe predict karo
val_predictions = model.predict(X_val)

# Accuracy nikalo
val_accuracy = accuracy_score(y_val, val_predictions)
print("Baseline Logistic Regression Validation Accuracy:", round(val_accuracy, 4))

Baseline Logistic Regression Validation Accuracy: 0.7918


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [26]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Fresh model banate hain (already-trained wala nahi)
cv_model = LogisticRegression(max_iter=1000, random_state=42)

# 5-Fold stratified cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(cv_model, X, y, cv=skf, scoring='accuracy')

print("Har fold ka accuracy:", cv_scores.round(4))
print("Mean accuracy:", round(cv_scores.mean(), 4))
print("Std deviation:", round(cv_scores.std(), 4))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Har fold ka accuracy: [0.7913 0.7924 0.7941 0.8003 0.7814]
Mean accuracy: 0.7919
Std deviation: 0.0061


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [27]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Sirf X_train se fit karo (seekhna), phir dono pe transform (apply) karo
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Model dobara train karo, ab scaled data pe
model_scaled = LogisticRegression(max_iter=1000, random_state=42)
model_scaled.fit(X_train_scaled, y_train)

val_predictions_scaled = model_scaled.predict(X_val_scaled)
val_accuracy_scaled = accuracy_score(y_val, val_predictions_scaled)
print("Scaled Logistic Regression Validation Accuracy:", round(val_accuracy_scaled, 4))

Scaled Logistic Regression Validation Accuracy: 0.7936


In [28]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators = 200,random_state = 42)
rf_cv_scores = cross_val_score(rf_model,X,y,cv=skf,scoring = 'accuracy')
print("Random Forest - Har fold ka accuracy:", rf_cv_scores.round(4))
print("Random Forest - Mean accuracy:", round(rf_cv_scores.mean(), 4))
print("Random Forest - Std deviation:", round(rf_cv_scores.std(), 4))

Random Forest - Har fold ka accuracy: [0.8114 0.7976 0.7982 0.8084 0.7808]
Random Forest - Mean accuracy: 0.7993
Random Forest - Std deviation: 0.0107


In [29]:

rf_model.fit(X, y)

importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

display(importance_df.head(15))

,Feature,Importance
8,CabinNum,0.160087
1,Age,0.125719
6,Spa,0.100102
3,RoomService,0.097814
7,VRDeck,0.096364
0,CryoSleep,0.084745
4,FoodCourt,0.081215
5,ShoppingMall,0.074812
9,GroupSize,0.030833
22,Side_S,0.021399


In [30]:

for df in [train_encoded, test_encoded]:
    df['TotalSpend'] = df[spend_cols].sum(axis=1)


X_new = train_encoded.drop(columns=['PassengerId', 'Transported'])


rf_model_v2 = RandomForestClassifier(n_estimators=200, random_state=42)
rf_cv_scores_v2 = cross_val_score(rf_model_v2, X_new, y, cv=skf, scoring='accuracy')

print("RF + TotalSpend - Mean accuracy:", round(rf_cv_scores_v2.mean(), 4))
print("RF + TotalSpend - Std deviation:", round(rf_cv_scores_v2.std(), 4))

RF + TotalSpend - Mean accuracy: 0.7996
RF + TotalSpend - Std deviation: 0.0092


In [31]:
from sklearn.metrics import confusion_matrix, classification_report

# Random Forest (with TotalSpend) ko X_train/X_val split pe train karte hain interpretation ke liye
X_train_new = X_train.copy()
X_val_new = X_val.copy()
X_train_new['TotalSpend'] = X_train_new[spend_cols].sum(axis=1)
X_val_new['TotalSpend'] = X_val_new[spend_cols].sum(axis=1)

rf_final = RandomForestClassifier(n_estimators=200, random_state=42)
rf_final.fit(X_train_new, y_train)
val_preds = rf_final.predict(X_val_new)

# Confusion matrix
cm = confusion_matrix(y_val, val_preds)
print("Confusion Matrix:\n", cm)

# Precision, Recall, F1
print("\n", classification_report(y_val, val_preds))

Confusion Matrix:
 [[727 136]
 [199 677]]

               precision    recall  f1-score   support

       False       0.79      0.84      0.81       863
        True       0.83      0.77      0.80       876

    accuracy                           0.81      1739
   macro avg       0.81      0.81      0.81      1739
weighted avg       0.81      0.81      0.81      1739



In [32]:

final_model = RandomForestClassifier(n_estimators=200, random_state=42)
final_model.fit(X_new, y)

# Step 2: Test data pe bhi exactl
X_test_final = test_encoded.drop(columns=['PassengerId'])

# Step 3: Column order match karna zaroori hai (train aur test ke columns same order mein hon)
X_test_final = X_test_final[X_new.columns]

# Step 4: Predict karo
test_predictions = final_model.predict(X_test_final)

# Step 5: Submission format banao
submission = pd.DataFrame({
    'PassengerId': test_encoded['PassengerId'],
    'Transported': test_predictions
})

# Step 6: Checks
print("Submission shape:", submission.shape)
print("Expected rows:", test.shape[0])
print("\nMissing predictions:", submission['Transported'].isnull().sum())
print("\nFirst 5 rows:\n", submission.head())

# Step 7: Save
submission.to_csv('submission_file.csv', index=False)
print("\nsubmission.csv saved!")

Submission shape: (4277, 2)
Expected rows: 4277

Missing predictions: 0

First 5 rows:
   PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01        False

submission.csv saved!


In [33]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier(n_estimators = 200,random_state = 42, eval_metric ='logloss')
xgb_cv_scores = cross_val_score(xgb_model, X_new,y, cv = skf, scoring ='accuracy')
print("XGBoost Every folder : ",xgb_cv_scores.round(4))
print("XGBoost - Mean accuracy :", round(xgb_cv_scores.mean(),4))
print("XGBoost - std deviation :", round(xgb_cv_scores.std(),4))

XGBoost Every folder :  [0.812  0.8085 0.7884 0.8049 0.7906]
XGBoost - Mean accuracy : 0.8009
XGBoost - std deviation : 0.0096


In [34]:
for depth in [3,4,6]:
    xgb_tuned =XGBClassifier(
        n_estimators = 200,
        max_depth = depth,
        random_state = 42,
        eval_metric = 'logloss'
    )
    scores = cross_val_score(xgb_tuned,X_new,y,cv = skf,scoring = 'accuracy')
    print(f"max_depth = {depth} -> Mean: {scores.mean() : .4f},Std : {scores.std(): .4f}")

max_depth = 3 -> Mean:  0.8073,Std :  0.0061
max_depth = 4 -> Mean:  0.8089,Std :  0.0084
max_depth = 6 -> Mean:  0.8009,Std :  0.0096


In [35]:
for lr in [0.05, 0.1, 0.3]:
    xgb_lr = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=lr,
        random_state=42,
        eval_metric='logloss'
    )
    scores = cross_val_score(xgb_lr, X_new, y, cv=skf, scoring='accuracy')
    print(f"learning_rate={lr} -> Mean: {scores.mean():.4f}, Std: {scores.std():.4f}")

learning_rate=0.05 -> Mean: 0.8093, Std: 0.0058
learning_rate=0.1 -> Mean: 0.8128, Std: 0.0060
learning_rate=0.3 -> Mean: 0.8089, Std: 0.0084


In [36]:
# Final tuned model: poore training data pe train
final_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
final_xgb.fit(X_new, y)

# Test data: same columns, same order
X_test_final = test_encoded.drop(columns=['PassengerId'])
X_test_final = X_test_final[X_new.columns]

# Predict aur submission banao
xgb_predictions = final_xgb.predict(X_test_final)

submission_xgb = pd.DataFrame({
    'PassengerId': test_encoded['PassengerId'],
    'Transported': xgb_predictions.astype(bool)
})

# Checks
print("Shape:", submission_xgb.shape, "| Expected rows:", test.shape[0])
print("Missing predictions:", submission_xgb['Transported'].isnull().sum())
print(submission_xgb['Transported'].value_counts(normalize=True).round(3))
print(submission_xgb.head())

submission_xgb.to_csv('submission_xgb.csv', index=False)
print("submission_xgb.csv saved!")

Shape: (4277, 2) | Expected rows: 4277
Missing predictions: 0
Transported
True     0.513
False    0.487
Name: proportion, dtype: float64
  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01         True
submission_xgb.csv saved!
